# PNAD Income Analysis Pipeline

This is the single executable notebook for the project. It does not implement statistical routines: all calculations are imported from the documented package under `src/pnad_income/`. The notebook executes the complete pipeline, explains each stage, and displays the resulting diagnostics, tables, and figures.

## 1. Configuration

By default the analysis reads the annual Parquet files in `dados_refined/`. Set `PNAD_DATABASE_PATH` only when a different database should be analyzed.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

from pnad_income.pipeline import PipelineConfig, pipeline_overview, run_pipeline
from pnad_income.plotting import (
    plot_ccdf_selected_years,
    plot_gini_evolution,
    plot_lorenz_curve,
    plot_measure_comparison,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATABASE_PATH = Path(os.environ.get("PNAD_DATABASE_PATH", "../dados_refined")).expanduser()
CONFIG = PipelineConfig(database_path=DATABASE_PATH, ccdf_base=1.05, start_year=1976, end_year=2025)
CONFIG

## 2. Load and validate the database

`run_pipeline` loads every annual file, infers missing year identifiers from filenames, validates the schema, attaches monetary metadata, and creates adjusted income variables. The overview below is the first reproducibility diagnostic.

In [ ]:
results = run_pipeline(CONFIG)
overview = pipeline_overview(results)
display(overview)
display(results.panel.head())

## 3. Annual descriptive statistics and inequality

For every available year, the pipeline computes the sample size, positive support, mean, median, sample standard deviation, and Gini coefficient. Adjusted measures are kept separate from nominal values.

In [ ]:
summary = results.summary
display(summary)
plot_gini_evolution(summary, value_col="income")
plt.show()

## 4. Lorenz curve

The Lorenz curve shows cumulative income share as a function of cumulative population share. The latest survey year is displayed below; the year can be changed without changing the analytical implementation.

In [ ]:
latest_year = results.years[-1]
plot_lorenz_curve(results.panel, latest_year, value_col="income")
plt.show()

## 5. Complementary cumulative distribution functions

The empirical CCDF is $\widehat{\overline F}(x)=N^{-1}\sum_i \mathbf{1}(X_i\ge x)$. Geometric thresholds start at the smallest strictly positive income, while finite zero-income observations remain in the normalization denominator. The table below contains nominal and adjusted annual CCDFs.

In [ ]:
ccdf = results.ccdf_nominal_adjusted
display(ccdf.head(20))

years = results.years
representative_years = sorted(set([years[0], years[len(years)//3], years[(2*len(years))//3], years[-1]]))
plot_ccdf_selected_years(ccdf, measure="income", years=representative_years, transform="loglog")
plt.show()

## 6. Nominal versus adjusted income

The monetary transformation changes the income scale while preserving the observation set. The latest year is compared below on log-log axes.

In [ ]:
plot_measure_comparison(ccdf, latest_year, measures=("income", "income_adj"), transform="loglog")
plt.show()

## 7. Habitual versus effective income

If the refined database contains `income_effective`, the package computes the second distribution and compares it with `income` using identical procedures. If that column is absent, the notebook reports the missing analytical input explicitly rather than fabricating the result.

In [ ]:
ccdf_effective = results.ccdf_habitual_effective
if ccdf_effective.empty:
    print("The current refined database does not contain usable income_effective observations.")
else:
    effective_years = sorted(ccdf_effective["year"].unique())
    display(ccdf_effective.head(20))
    plot_measure_comparison(
        ccdf_effective,
        int(effective_years[-1]),
        measures=("income", "income_effective"),
        transform="loglog",
    )
    plt.show()

## 8. Final data-quality diagnostics

The final table records missingness and numerical support of every central analytical measure. It should be checked after any update to the refined database before using generated figures or tables in a report or manuscript.

In [ ]:
diagnostic_columns = [c for c in ("income", "income_adj", "income_effective", "income_effective_adj") if c in results.panel.columns]
diagnostics = pd.DataFrame({
    "column": diagnostic_columns,
    "non_missing": [int(results.panel[c].notna().sum()) for c in diagnostic_columns],
    "missing": [int(results.panel[c].isna().sum()) for c in diagnostic_columns],
    "minimum": [results.panel[c].min() for c in diagnostic_columns],
    "maximum": [results.panel[c].max() for c in diagnostic_columns],
})
display(diagnostics)

## 9. Result objects

The reusable outputs are `results.panel`, `results.summary`, `results.ccdf_nominal_adjusted`, and `results.ccdf_habitual_effective`. Subsequent modeling should start from these validated objects rather than reimplementing preprocessing in notebook cells.